# Lesson 02 - Microsoft Agent Framework tyrinėjimas

**Microsoft Agent Framework (MAF)** yra vieninga sistema AI agentams kurti. Ji suteikia aiškią, sudedamą architektūrą su keturiais pagrindiniais komponentais:

- **Klientas** – jungiasi prie AI modelio galo taško ir tvarko ryšį
- **Agentas** – apgaubia klientą su instrukcijomis ir įrankių apibrėžimais
- **Įrankiai** – išplečia agento galimybes su pasirinktinių funkcijų, kurias modelis gali kviesti, rinkiniais
- **Sesija** – palaiko pokalbio istoriją daugkartiniams apsikeitimams

Šioje pamokoje sukursime **kelionių užsakymo agentą**, kuris naudos šias sąvokas tikrinant kelionės paskirties prieinamumą.


## Įrengimas


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Supratimas apie Agentų platformos architektūrą

„Microsoft Agent Framework“ naudoja sluoksniuotą architektūrą:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klientas** – `AzureAIProjectAgentProvider` jungiasi prie „Azure OpenAI“ diegimo. Jis tvarko autentifikavimą, užklausų formatavimą ir atsakymų analizę.
2. **Agentas** – Sukuriamas iš kliento per `provider.create_agent()`, agentas apjungia modelio prieigą su instrukcijomis (sistemos nurodymu) ir įrankiais.
3. **Įrankiai** – Python funkcijos, pažymėtos `@tool`, kurias agentas gali iškviesti veiksmams atlikti arba duomenims gauti.
4. **Seansas** – `AgentSession` objektas (sukurtas per `agent.create_session()`), kuris saugo pokalbio istoriją, leidžiantį vykdyti kelių apverstų dialogą, kur agentas prisimena ankstesnį kontekstą.

Sukurkime kiekvieną sluoksnį žingsnis po žingsnio.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Įrankių pridėjimas naudojant @tool dekoratorių

Įrankiai leidžia agentams imtis veiksmų, neapsiribojant tekstų generavimu. `@tool` dekoratorius paverčia įprastą Python funkciją į kažką, ką agentas gali iškviesti.

Svarbiausi punktai:
- Naudokite `Annotated[type, "description"]`, kad modelis suprastų kiekvieną parametrą.
- Docstring tampa įrankio aprašymu, kurį mato modelis.
- `approval_mode="never_require"` reiškia, kad įrankis vykdomas automatiškai be vartotojo patvirtinimo.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agento kūrimas su įrankiais

Dabar sujungiame klientą, instrukcijas ir įrankius į agentą. `instructions` veikia kaip sistemos raginimas — jie apibrėžia agente asmenybę ir elgesį.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Daugiauapimčiai Pokalbiai su Sesijomis

`AgentSession` (sukuriama per `agent.create_session()`) seka visas žinutes pokalbyje. Perdami tą pačią sesiją kiekvienam `agent.run()` kvietimui, agentas turi prieigą prie visos pokalbio istorijos ir gali atsiremti į ankstesnes žinutes.

Perduodame `tools=[check_destination_availability]`, kad agentas galėtų kiekvieno ėjimo metu pasinaudoti mūsų prieinamumo tikrintuvu.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Santrauka

Šiame pamokoje tyrinėjote keturis Microsoft Agent Framework pamatinius akmenis:

| Sąvoka | Ką išmokote |
|---------|------------------|
| **Klientas** | `AzureAIProjectAgentProvider` jungiasi prie Azure OpenAI naudodamas kredencialais pagrįstą autentifikavimą |
| **Agentas** | `provider.create_agent()` sujungia modelio ryšį su instrukcijomis ir pavadinimu |
| **Įrankiai** | `@tool` dekoratorius leidžia agentui iškvietinėti Python funkcijas |
| **Seansas** | `agent.create_session()` palaiko pokalbio istoriją per kelis ciklus |

Šie statybiniai blokai sujungiami, kad sukurtų agentus, kurie gali vesti natūralias diskusijas, iškviesti išorines funkcijas ir palaikyti kontekstą – tai pagrindas pažangesniems agentiniams modeliams vėlesnėse pamokose.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas gimtąja kalba laikomas autoritetingu šaltiniu. Kritinei informacijai rekomenduojamas profesionalus žmogaus vertimas. Mes neatsakome už bet kokius nesusipratimus ar neteisingus interpretavimus, kilusius naudojant šį vertimą.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
